In [ ]:
# parameters
# BINDER_FAST: set N=6 for fast cloud execution
N = 20          # Hilbert space truncation
Ej_cpb = 1.0   # Josephson energy for Cooper-pair box (GHz), Ej/Ec ~ 0.2
Ec_cpb = 5.0   # Charging energy for Cooper-pair box (GHz)
Ej_trans = 50.0  # Josephson energy in transmon regime (GHz), Ej/Ec ~ 250
Ec_trans = 0.2   # Charging energy in transmon regime (GHz)


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.hamiltonians.transmon import transmon_hamiltonian
from bosonic_gates.hamiltonians.harmonic_oscillator import resonator_hamiltonian


## Module 1a: From the LC Oscillator to the Josephson Junction

**Learning objectives:**
- Quantize the LC circuit and identify the conjugate variables $\hat{\phi}$ and $\hat{Q}$
- Understand why replacing the linear inductor with a Josephson junction creates a nonlinear, anharmonic spectrum
- Write down the Cooper-pair box Hamiltonian and identify the two competing energy scales $E_J$ and $E_C$
- Explain charge dispersion and why large $E_J/E_C$ suppresses it (the transmon limit)

---

**Sections:** [1 LC Circuit](#sec1) · [2 Josephson Junction](#sec2) · [3 Cooper-Pair Box](#sec3) · [4 Ej/Ec Ratio](#sec4)


<a id="sec1"></a>
## 1  The LC Oscillator

A lumped-element LC circuit is the simplest microwave resonator.
Its classical Lagrangian uses the node flux $\Phi$ — the time-integral of the voltage — as the generalized coordinate:

$$\mathcal{L} = \frac{C}{2}\dot{\Phi}^2 - \frac{\Phi^2}{2L}.$$

The conjugate momentum is the charge $Q = \partial\mathcal{L}/\partial\dot{\Phi} = C\dot{\Phi}$,
so the classical Hamiltonian is

$$H = \frac{Q^2}{2C} + \frac{\Phi^2}{2L}.$$

**Canonical quantization.** Promote $\Phi$ and $Q$ to operators satisfying

$$[\hat{\Phi}, \hat{Q}] = i\hbar.$$

This is structurally identical to the position-momentum commutator $[\hat{x}, \hat{p}] = i\hbar$.
Define dimensionless ladder operators

$$\hat{a} = \frac{1}{\sqrt{2\hbar Z_r}}\left(Z_r\hat{Q} - i\hat{\Phi}\right), \qquad
\hat{a}^\dagger = \frac{1}{\sqrt{2\hbar Z_r}}\left(Z_r\hat{Q} + i\hat{\Phi}\right),$$

where $Z_r = \sqrt{L/C}$ is the characteristic impedance.
In terms of $\hat{a}$ and $\hat{a}^\dagger$, the Hamiltonian takes the canonical form:

$$\hat{H}_{LC} = \hbar\omega_r\!\left(\hat{a}^\dagger\hat{a} + \frac{1}{2}\right), \qquad \omega_r = \frac{1}{\sqrt{LC}}.$$

The energy levels are **equally spaced**: $E_n = \hbar\omega_r(n+\tfrac{1}{2})$.
This means every transition $|n\rangle \to |n+1\rangle$ resonates at the same frequency $\omega_r$ —
a classical drive cannot selectively address a single rung of the ladder.
To make a qubit we need an **anharmonic** energy spectrum.

*We work in natural units with $\hbar = 1$ throughout.*

> **Lab note.** *In a coplanar waveguide (CPW) readout resonator, $\omega_r/2\pi$ is typically 4–8 GHz.
> You measure it as the dip in $S_{21}$ on a VNA.
> The zero-point fluctuation amplitude is $\Phi_{\rm zpf} = \sqrt{\hbar Z_r/2}$;
> for $Z_r = 50\,\Omega$ and $\omega_r/2\pi = 6\,\text{GHz}$ this is about $1\,\mu\Phi_0$,
> roughly $10^{-9}$ of the superconducting flux quantum.*


In [ ]:
# LC resonator energy levels
omega_r_demo = 2 * np.pi * 6.0  # 6 GHz resonator
H_res = resonator_hamiltonian(w=omega_r_demo, M=N)
evals_res = H_res.eigenenergies()

fig, ax = plt.subplots(figsize=(3.5, 5))
for n, E in enumerate(evals_res[:8]):
    ax.axhline(E / (2 * np.pi), xmin=0.15, xmax=0.75, lw=2.5, color="steelblue")
    ax.text(0.78, E / (2 * np.pi), rf"$|{n}\rangle$", va="center", fontsize=10,
            transform=ax.get_yaxis_transform())

ax.set_ylabel(r"Energy $/ (2\pi)$ (GHz)")
ax.set_xticks([])
ax.set_title(r"LC resonator: equally spaced levels")
ax.spines[["top", "right", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()

print("Transition frequencies (GHz):")
for n in range(5):
    spacing = (evals_res[n + 1] - evals_res[n]) / (2 * np.pi)
    print(f"  omega_{n},{n+1} / 2pi = {spacing:.4f} GHz")


<a id="sec2"></a>
## 2  The Josephson Junction: Replacing the Linear Inductor

A Josephson junction (JJ) is a thin insulating barrier between two superconductors.
Cooper pairs tunnel coherently through the barrier, governed by two Josephson relations:

$$I = I_c \sin(\hat{\phi}), \qquad \dot{\hat{\phi}} = \frac{2e}{\hbar}V,$$

where $I_c$ is the critical current and $\hat{\phi} = 2\pi\hat{\Phi}/\Phi_0$ is the dimensionless gauge-invariant phase difference
($\Phi_0 = h/2e$ is the flux quantum).

**Josephson inductance.** Differentiating the first Josephson relation gives a phase-dependent inductance:

$$L_J(\phi) = \frac{\Phi_0}{2\pi I_c \cos(\phi)}.$$

At small phase (linear regime) $L_J(0) = \Phi_0/(2\pi I_c)$ — this is the usual junction inductance.
But unlike a linear inductor, the potential energy is **nonlinear** (cosine rather than parabolic):

$$U_J(\hat{\phi}) = -E_J\cos(\hat{\phi}), \qquad E_J = \frac{\hbar I_c}{2e} = \frac{\Phi_0 I_c}{2\pi}.$$

Expanding the cosine:

$$U_J(\hat{\phi}) = -E_J\!\left[1 - \frac{\hat{\phi}^2}{2} + \frac{\hat{\phi}^4}{24} - \cdots\right]
= -E_J + \frac{E_J}{2}\hat{\phi}^2 - \frac{E_J}{24}\hat{\phi}^4 + \cdots$$

The $\hat{\phi}^2$ term acts like a linear inductance with $L_J = \hbar^2/4e^2 E_J$.
The $\hat{\phi}^4$ correction breaks the equal level spacing — this is the **anharmonicity** we need to make a qubit.

> **Lab note.** *$E_J$ is tunable if the junction is replaced by a SQUID (two junctions in a loop):
> threading a flux $\Phi_{ext}$ through the loop gives an effective $E_J^{\rm eff} = E_J^{\rm max}|\cos(\pi\Phi_{ext}/\Phi_0)|$.
> This is how flux-tunable transmons and SNAILs work — you modulate $E_J$ in situ with an on-chip coil.
> The Josephson energy of a typical Al/AlO$_x$/Al junction ranges from 1 to 100 GHz.*


<a id="sec3"></a>
## 3  The Cooper-Pair Box

Connect a Josephson junction to a small superconducting island coupled to a gate voltage $V_g$ through a gate capacitor $C_g$.
The island has total capacitance $C_\Sigma = C_J + C_g$.
The dimensionless gate charge $n_g = C_g V_g / 2e$ controls how many Cooper pairs are energetically favorable on the island.

The Hamiltonian is the **Cooper-pair box (CPB)**:

$$\hat{H}_{\rm CPB} = 4E_C(\hat{n} - n_g)^2 - E_J\cos(\hat{\phi}),$$

where:
- $\hat{n}$ is the Cooper-pair number operator (integer eigenvalues)
- $\hat{\phi}$ is the conjugate phase operator, $[\hat{\phi}, \hat{n}] = i$
- $E_C = e^2 / 2C_\Sigma$ is the **charging energy** — the electrostatic cost of adding one extra electron charge
- $4E_C$ is the cost of adding one Cooper pair ($2e$)
- $E_J$ is the **Josephson energy** — the tunnel coupling that favors phase coherence

**Charge dispersion in the CPB regime ($E_J/E_C \sim 1$).** When $E_J \ll E_C$, the charging energy dominates.
The energy levels depend strongly on $n_g$ — the qubit frequency shifts by order $E_C$ as $n_g$ varies by 1.
This sensitivity to offset charges (environmental charge noise) causes rapid dephasing,
making the CPB impractical as a qubit in the laboratory.

The plot below shows this strong $n_g$-dependence clearly.

> **Lab note.** *The first CPB qubit was demonstrated by Nakamura et al. (Nature 1999).
> The charge sweet spot at $n_g = 1/2$ is the degeneracy point where two charge states are degenerate
> and the first-order sensitivity to charge noise vanishes.
> However, the protection only works at exactly $n_g = 0.5$, so even small charge drifts detune the qubit.
> The transmon (Section 4) eliminates this problem at the cost of reduced anharmonicity.*


In [ ]:
# Charge dispersion plot: energy levels vs ng for CPB regime
ng_vals = np.linspace(0, 1, 60)
n_levels = 6
levels = np.zeros((n_levels, len(ng_vals)))

for i, ng in enumerate(ng_vals):
    H = transmon_hamiltonian(Ej_cpb, Ec_cpb, N, ng=ng)
    evals = H.eigenenergies()[:n_levels]
    levels[:, i] = evals - evals[0]  # zero at ground state

colors = plt.cm.viridis(np.linspace(0.1, 0.9, n_levels))

fig, ax = plt.subplots(figsize=(7, 5))
for m in range(n_levels):
    ax.plot(ng_vals, levels[m] / (2 * np.pi), color=colors[m],
            lw=2, label=rf"$m = {m}$")

ax.set_xlabel(r"Gate charge $n_g$", fontsize=12)
ax.set_ylabel(r"$E_m(n_g) - E_0(n_g)$ $\;/ 2\pi$ (GHz)", fontsize=12)
ax.set_title(
    rf"Cooper-pair box: $E_J/E_C = {Ej_cpb/Ec_cpb:.2f}$ — strong charge dispersion",
    fontsize=11
)
ax.legend(loc="upper right", fontsize=9)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

# Charge dispersion: peak-to-peak variation of the qubit frequency
disp = (levels[1].max() - levels[1].min()) / (2 * np.pi)
print(f"CPB qubit charge dispersion (peak-to-peak): {disp:.3f} GHz")
print(f"  Ej = {Ej_cpb} GHz,  Ec = {Ec_cpb} GHz,  Ej/Ec = {Ej_cpb/Ec_cpb:.2f}")


<a id="sec4"></a>
## 4  The Ej/Ec Ratio: CPB vs Transmon

The two extremes of the Cooper-pair box are controlled by the ratio $E_J/E_C$:

| Regime | $E_J/E_C$ | Character |
|--------|----------|----------|
| **CPB** | $\lesssim 1$ | Charge qubit; large charge dispersion; sensitive to $1/f$ charge noise |
| **Transmon** | $\gtrsim 40$ | Flux qubit; charge dispersion exponentially suppressed |

**Transmon as a weakly anharmonic oscillator.**
Koch et al. (PRA 2007) showed that for large $E_J/E_C$ the CPB approaches a harmonic oscillator,
but retains a small **anharmonicity**

$$\alpha \equiv (E_{12} - E_{01}) \approx -E_C,$$

where $E_{mn} = \omega_{mn}/2\pi$ is the $|m\rangle \to |n\rangle$ transition frequency.
The negative sign means $\omega_{12} < \omega_{01}$: the second transition is *lower* than the first.
Typical transmons have $|\alpha|/2\pi \approx 100$–$300\,\text{MHz}$.

**Charge dispersion suppression.** The charge dispersion of the $m$-th level scales as

$$\epsilon_m \propto \left(\frac{E_J}{E_C}\right)^{1/2}\exp\!\left(-\sqrt{8E_J/E_C}\right),$$

which is exponentially small in $\sqrt{E_J/E_C}$.
For $E_J/E_C = 50$, the dispersion is already $\sim 10^{-5}$ smaller than for $E_J/E_C = 1$.

**Trade-off.** Reducing charge sensitivity also reduces anharmonicity since both scale with $E_C$.
A typical transmon operating point of $E_J/E_C \approx 50$–$100$ balances
gate speed ($\sim 1/|\alpha|$) against charge noise sensitivity.

> **Lab note.** *In practice, $E_C/h \approx 200$–$350\,\text{MHz}$ and $E_J/h \approx 10$–$50\,\text{GHz}$
> for aluminium-based transmons.
> The qubit frequency $\omega_{01}/2\pi \approx \sqrt{8E_JE_C} - E_C$ is set by the junction critical current,
> which is fixed by the junction area during fabrication.
> Flux-tunable SNAILs and asymmetric-SQUID transmons allow in-situ tuning over a few GHz.*


In [ ]:
# Sweep Ej/Ec ratio: qubit frequency and anharmonicity
ratios = np.logspace(0, 2, 40)  # 1 to 100
Ec_fixed = 0.2  # GHz (fixed)

omega01_list = []
alpha_list = []

for ratio in ratios:
    Ej = ratio * Ec_fixed
    H = transmon_hamiltonian(Ej, Ec_fixed, N)
    evals = H.eigenenergies()[:4]
    omega01 = (evals[1] - evals[0]) / (2 * np.pi)   # GHz
    alpha   = ((evals[2] - evals[1]) - (evals[1] - evals[0])) / (2 * np.pi)  # GHz
    omega01_list.append(omega01)
    alpha_list.append(alpha)

omega01_arr = np.array(omega01_list)
alpha_arr   = np.array(alpha_list)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogx(ratios, omega01_arr, lw=2.5, color="steelblue")
# Analytic approximation: sqrt(8*Ej*Ec) - Ec
omega01_approx = (np.sqrt(8 * ratios * Ec_fixed**2) - Ec_fixed)
axes[0].semilogx(ratios, omega01_approx, "--", lw=1.5, color="tomato",
                 label=r"$\sqrt{8E_JE_C} - E_C$")
axes[0].set_xlabel(r"$E_J / E_C$", fontsize=12)
axes[0].set_ylabel(r"$\omega_{01} / 2\pi$ (GHz)", fontsize=12)
axes[0].set_title(r"Qubit frequency vs $E_J/E_C$")
axes[0].legend()

axes[1].semilogx(ratios, alpha_arr, lw=2.5, color="darkorange", label=r"Numerical $\alpha$")
axes[1].axhline(-Ec_fixed, ls="--", color="gray", lw=1.5, label=rf"$-E_C = {-Ec_fixed}$ GHz")
axes[1].set_xlabel(r"$E_J / E_C$", fontsize=12)
axes[1].set_ylabel(r"Anharmonicity $\alpha / 2\pi$ (GHz)", fontsize=12)
axes[1].set_title(r"Anharmonicity vs $E_J/E_C$")
axes[1].legend()

plt.suptitle(rf"$E_C = {Ec_fixed}$ GHz fixed, $N = {N}$", fontsize=11)
plt.tight_layout()
plt.show()

# Print transmon limit values
idx_trans = np.argmin(np.abs(ratios - 50))
print(f"At Ej/Ec = {ratios[idx_trans]:.1f}:")
print(f"  omega_01 / 2pi = {omega01_arr[idx_trans]:.4f} GHz")
print(f"  anharmonicity alpha / 2pi = {alpha_arr[idx_trans]:.4f} GHz")
print(f"  -Ec = {-Ec_fixed:.4f} GHz  (transmon limit)")
